In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

forest_file = 'Bird_Monitoring_Data_FOREST.XLSX'
grass_file  = 'Bird_Monitoring_Data_GRASSLAND.XLSX'

# Expected columns (we will standardize everything to this list)
EXPECTED_COLUMNS = [
    'Admin_Unit_Code', 'Sub_Unit_Code', 'Site_Name', 'Plot_Name', 'Location_Type',
    'Year', 'Date', 'Start_Time', 'End_Time', 'Observer', 'Visit', 'Interval_Length',
    'ID_Method', 'Distance', 'Flyover_Observed', 'Sex', 'Common_Name', 'Scientific_Name',
    'AcceptedTSN', 'NPSTaxonCode', 'AOU_Code', 'PIF_Watchlist_Status',
    'Regional_Stewardship_Status', 'Temperature', 'Humidity', 'Sky', 'Wind',
    'Disturbance', 'Initial_Three_Min_Cnt', 'Habitat', 'Source_Sheet'
]

def validate_and_clean_sheet(df: pd.DataFrame, sheet_name: str, habitat: str) -> pd.DataFrame:
    """Clean + Validate ONE sheet. Returns cleaned df or empty df if invalid."""
    
    print(f"Processing sheet: {sheet_name} ({habitat}) - Rows: {len(df)}")
    
    # 1. Skip empty sheets (only header row)
    if len(df) <= 1:
        print(f"   → SKIPPED (empty sheet)")
        return pd.DataFrame()
    
    # 2. Standardize column names (Grassland has slight differences)
    rename_map = {
        'TaxonCode': 'NPSTaxonCode',      # Grassland uses TaxonCode
        'Previously_Obs': None            # We will drop this column (not in Forest)
    }
    df = df.rename(columns=rename_map)
    
    # 3. Add metadata columns
    df['Habitat'] = habitat
    df['Source_Sheet'] = sheet_name
    
    # 4. Fix Date (Excel serial date → proper datetime)
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], unit='D', origin='1899-12-30', errors='coerce')
    
    # 5. Convert Start_Time / End_Time (fractions of day → timedelta)
    for col in ['Start_Time', 'End_Time']:
        if col in df.columns:
            df[col] = pd.to_timedelta(df[col], unit='D', errors='coerce')
    
    # 6. Convert numeric columns
    numeric_cols = ['Year', 'Temperature', 'Humidity', 'Initial_Three_Min_Cnt', 'Visit']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 7. Standardize boolean columns
    bool_cols = ['Flyover_Observed', 'PIF_Watchlist_Status', 'Regional_Stewardship_Status']
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].map({True: True, False: False, 'true': True, 'false': False, 
                                 'TRUE': True, 'FALSE': False}).fillna(False)
    
    # 8. Fill missing critical values
    df['Sex'] = df['Sex'].fillna('Undetermined')
    df['Distance'] = df['Distance'].fillna('Unknown')
    df['ID_Method'] = df['ID_Method'].fillna('Unknown')
    
    # 9. Create useful derived columns
    def interval_to_minutes(x):
        if pd.isna(x): return np.nan
        x = str(x).lower()
        if '0-2.5' in x: return 2.5
        if '2.5-5' in x: return 5.0
        if '5-7.5' in x: return 7.5
        if '7.5-10' in x: return 10.0
        return np.nan
    
    df['Interval_Min'] = df['Interval_Length'].apply(interval_to_minutes)
    
    # 10. Final column ordering + missing columns filled with NaN
    for col in EXPECTED_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan
    df = df[EXPECTED_COLUMNS]
    
    # Validation summary
    print(f"   → VALIDATED: {len(df)} rows | Species: {df['Scientific_Name'].nunique()}")
    
    return df

cleaned_sheets = []

# Process Forest file
print("\n PROCESSING FOREST FILE...")
xl_forest = pd.ExcelFile(forest_file)
for sheet in xl_forest.sheet_names:
    df = pd.read_excel(xl_forest, sheet_name=sheet, header=0)
    cleaned = validate_and_clean_sheet(df, sheet, 'Forest')
    if not cleaned.empty:
        cleaned_sheets.append(cleaned)

# Process Grassland file
print("\n PROCESSING GRASSLAND FILE...")
xl_grass = pd.ExcelFile(grass_file)
for sheet in xl_grass.sheet_names:
    df = pd.read_excel(xl_grass, sheet_name=sheet, header=0)
    cleaned = validate_and_clean_sheet(df, sheet, 'Grassland')
    if not cleaned.empty:
        cleaned_sheets.append(cleaned)

final_df = pd.concat(cleaned_sheets, ignore_index=True)

print("\n FINAL DATASET CREATED!")
print(f"Total rows      : {len(final_df):,}")
print(f"Forest rows     : {len(final_df[final_df['Habitat']=='Forest']):,}")
print(f"Grassland rows  : {len(final_df[final_df['Habitat']=='Grassland']):,}")
print(f"Unique species  : {final_df['Scientific_Name'].nunique():,}")
print(f"Admin units     : {final_df['Admin_Unit_Code'].nunique()}")

# Save cleaned dataset
final_df.to_csv('Cleaned_Bird_Observation_Data.csv', index=False)
final_df.to_excel('Cleaned_Bird_Observation_Data.xlsx', index=False)

print("\n Files saved:")
print("   • Cleaned_Bird_Observation_Data.csv")
print("   • Cleaned_Bird_Observation_Data.xlsx")